# Meeting 3: test

Tests the best run of each mode, named in `IMG2IMG_RUN` and `IP_RUN`, on the test photos and pushes the results to GitHub. The Meeting 1 baselines and `m2-base` are already tested; compare against their `test-results.json`. Weights are restored from Google Drive, and the notebook stops if any are missing.

Run only after both runs are chosen on validation and written down in `notebooks/meeting3.ipynb`, and never change them afterwards.

In [1]:
import subprocess, sys
from getpass import getpass
from pathlib import Path
BRANCH = "excercises/3"  # the branch holding colab/ and runs/
REPO = Path("/content/dane_syntetyczne")
if not (REPO / ".git").exists():
    t = getpass("GitHub token (hidden): ")
    subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, f"https://{t}@github.com/tmsz/dane_syntetyczne.git", str(REPO)], check=True)
if subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", BRANCH]).returncode:
    raise RuntimeError(f"Branch {BRANCH!r} is not on GitHub. On the Mac: git push -u origin {BRANCH}")
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "--quiet", "origin", BRANCH], check=True)
sys.path.insert(0, str(REPO / "colab"))
import colab_tools as ct
import importlib
importlib.reload(ct)  # re-running this cell picks up pulled changes
ct.authenticate()  # Drive + GitHub prompts now, before any download or training

Mounted at /content/drive
✅ Google Drive mounted, GitHub push access OK


In [2]:
IMG2IMG_RUN = "m3-strength-0.9"  # the img2img run kept after validation (notebooks/meeting3.ipynb, 6.1); never change after testing
IP_RUN = "m3-ip-0.3"             # the IP-Adapter run kept after validation (notebooks/meeting3.ipynb, 6.1); never change after testing

IMG2IMG_CANDIDATES = ['m3-strength-0.3', 'm3-strength-0.5', 'm3-strength-0.7', 'm3-strength-0.9']
IP_CANDIDATES = ['m3-ip-0.3', 'm3-ip-0.5', 'm3-ip-0.7', 'm3-ip-1.0']
assert IMG2IMG_RUN in IMG2IMG_CANDIDATES, f"set IMG2IMG_RUN to one of {IMG2IMG_CANDIDATES}"
assert IP_RUN in IP_CANDIDATES, f"set IP_RUN to one of {IP_CANDIDATES}"

ct.setup()  # pull, install, download data, check GPU
for run in [IMG2IMG_RUN, IP_RUN]:
    ct.restore_models(run)  # model.pt <- Drive
    ct.test_run(run)
ct.check_devices([IMG2IMG_RUN, IP_RUN])
ct.save_to_github(f"Meeting 3: test {IMG2IMG_RUN} and {IP_RUN}")

$ git pull --ff-only --quiet
$ /usr/bin/python3 -m pip install --quiet -e .


100%|██████████| 792M/792M [00:31<00:00, 24.8MB/s] 
100%|██████████| 19.2M/19.2M [00:02<00:00, 8.81MB/s]


Data: 905 course photos in ./data
GPU: Tesla T4
m3-strength-0.9: 5 models restored from Drive

=== test m3-strength-0.9 ===
$ synth-test --run m3-strength-0.9 --device cuda
m3-strength-0.9: 5 saved models on the test photos, cuda
  seed 0: Egyptian Mau recall 0.633, macro-F1 0.881
  seed 1: Egyptian Mau recall 0.633, macro-F1 0.885
  seed 2: Egyptian Mau recall 0.567, macro-F1 0.871
  seed 3: Egyptian Mau recall 0.600, macro-F1 0.877
  seed 4: Egyptian Mau recall 0.733, macro-F1 0.870
Test results of every seed are in /content/dane_syntetyczne/runs/m3-strength-0.9/test-results.json
m3-ip-0.3: 5 models restored from Drive

=== test m3-ip-0.3 ===
$ synth-test --run m3-ip-0.3 --device cuda
m3-ip-0.3: 5 saved models on the test photos, cuda
  seed 0: Egyptian Mau recall 0.667, macro-F1 0.886
  seed 1: Egyptian Mau recall 0.733, macro-F1 0.903
  seed 2: Egyptian Mau recall 0.633, macro-F1 0.882
  seed 3: Egyptian Mau recall 0.667, macro-F1 0.886
  seed 4: Egyptian Mau recall 0.733, macro-F1

### Close the Colab server

Runs only if everything above succeeded (Run all stops at the first error, so a failed session stays open to debug). Results are already on GitHub and Drive, so the runtime is released to stop using GPU quota. Set `CLOSE_SERVER = False` to keep it.

In [3]:
CLOSE_SERVER = True  # release the GPU runtime once everything is saved

if CLOSE_SERVER:
    from google.colab import runtime
    print("✅ All saved to GitHub and Drive. Closing the Colab server.")
    runtime.unassign()  # disconnects and deletes this runtime

✅ All saved to GitHub and Drive. Closing the Colab server.
